# Crust failure rate interpolator

This notebook creates an interpolator function for the evolution of the crust failure rate due to magnetic stresses as a function of time. The interpolator serves as a function that, given the age and the initial magnetic field of a neutron star, gives as output the failure rate at that age. The corresponding failure rates are obtained from magneto-thermal simulations for different values of the initial magnetic field. For more details on the implementation of the crust failures in the magneto-thermal 2D code see [Dehman et al. (2020)](https://ui.adsabs.harvard.edu/abs/2020ApJ...902L..32D/abstract).

The original simulation files contain information on the time at which each failure happens and the corresponding magnetic energy released. Depending on the initial magnetic field configuration and magnetic energy in the crust, failure events might not occur below an initial poloidal dipolar magnetic field value as the magnetic stresses are not strong enough to cause any failures. Therefore, for some values of the initial magnetic field, the files containing the failures could be empty. Since we have only sets of failures for initial magnetic field values of $10^{12}$, $10^{13}$, $10^{14}$, $10^{15}$ and $5 \times 10^{15}$ G, we need to interpolate between them to obtain the expected failures rate for any given age and initial magnetic field.

Note that the failure rate estimated from the magneto-thermal model is computed by assuming that the mechanic stresses in the crust are reset after every failure event, while the magnetic stresses continue to build up, i.e., the magnetic field remains tangled (see [Dehman et al. 2020](https://ui.adsabs.harvard.edu/abs/2020ApJ...902L..32D/abstract) for more details). This might lead to an overestimation of the number of failure events.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import pathlib
import pickle
from scipy import interpolate
from scipy.interpolate import UnivariateSpline
from scipy.interpolate import make_interp_spline
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

import utilities.plot_settings
from mlpoppyns.simulator.config_simulator import cfg

## Load the results from the magneto-thermal simulations

In [ ]:
base_path = pathlib.Path("../../")

# We distinguish between the various different B-field simulation models.
if cfg["magneto-thermal_model"] == "BSk24_dip-tor_heavy":
    simB12_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_50-50_1e12_H.d"
    )
    simB13_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_50-50_1e13_H.d"
    )
    simB14_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_50-50_1e14_H.d"
    )
    simB15_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_50-50_1e15_H.d"
    )
    simB5e15_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_50-50_5e15_H.d"
    )

    # Define file paths in a dictionary.
    file_paths = {
        "1e12": simB12_path,
        "1e13": simB13_path,
        "1e14": simB14_path,
        "1e15": simB15_path,
        "5e15": simB5e15_path,
    }

    # Define an array with the log10 of the initial magnetic field 
    # values for the different failure event sets.
    log_B0 = np.array([12, 13, 14, 15, np.log10(5.0e15)])

elif cfg["magneto-thermal_model"] == "BSk24_dip-tor_light":
    simB12_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_50-50_1e12_L.d"
    )
    simB13_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_50-50_1e13_L.d"
    )
    simB14_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_50-50_1e14_L.d"
    )
    simB15_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_50-50_1e15_L.d"
    )
    simB5e15_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_50-50_5e15_L.d"
    )

    # Define file paths in a dictionary.
    file_paths = {
        "1e12": simB12_path,
        "1e13": simB13_path,
        "1e14": simB14_path,
        "1e15": simB15_path,
        "5e15": simB5e15_path,
    }

    # Define an array with the log10 of the initial magnetic field 
    # values for the different failure event sets.
    log_B0 = np.array([12, 13, 14, 15, np.log10(5.0e15)])

elif cfg["magneto-thermal_model"] == "BSk24_multi_heavy":
    simB12_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_multi_1e12_H.d"
    )
    simB13_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_multi_1e13_H.d"
    )
    simB14_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_multi_1e14_H.d"
    )
    simB15_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_multi_1e15_H.d"
    )
    simB5e15_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_multi_5e15_H.d"
    )

    # Define file paths in a dictionary.
    # For this configuration, the field of 5e15 G is too 
    # computationally expensive as the number of failures is too high.
    # We therefore remove it from our analysis.
    file_paths = {
        "1e12": simB12_path,
        "1e13": simB13_path,
        "1e14": simB14_path,
        "1e15": simB15_path,
    }

    # Define an array with the log10 of the initial magnetic field 
    # values for the different failure event sets.
    log_B0 = np.array([12, 13, 14, 15])

elif cfg["magneto-thermal_model"] == "BSk24_multi_light":
    simB12_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_multi_1e12_L.d"
    )
    simB13_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_multi_1e13_L.d"
    )
    simB14_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_multi_1e14_L.d"
    )
    simB15_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_multi_1e15_L.d"
    )
    simB5e15_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "failures_multi_5e15_L.d"
    )

    # Define file paths in a dictionary.
    # For this configuration, the field of 5e15 G is too 
    # computationally expensive as the number of failures is too high.
    # We therefore remove it from our analysis.
    file_paths = {
        "1e12": simB12_path,
        "1e13": simB13_path,
        "1e14": simB14_path,
        "1e15": simB15_path,
    }

    # Define an array with the log10 of the initial magnetic field 
    # values for the different failure event sets.
    log_B0 = np.array([12, 13, 14, 15])

else:
    raise ValueError(
        "The specified magneto-thermal model is not supported or the chosen option does not provide information on crust failures."
    )

The files contain the following columns:
- the time in [yr] when the failure event occurs;
- the total magnetic energy in [erg] dissipated during the failure;
- the position in polar coordinates ($\theta$ in [rad] and $r$ in [km]) where the failure event occurs;
- the total volume in [cm$^3$] of the crust affected by the failure;
- the timestep in [yr] for the magnetic field evolution that is used in the simulation.

In [ ]:
# Define names of columns with relevant information.
columns = ["time", "energy", "theta", "radius", "volume", "timestep"]

# Load DataFrames safely into a dictionary.
dfs = {}
for key, path in file_paths.items():
    # Check if the files are empty, i.e., contain no failure events.
    if os.path.getsize(path) > 0:
        df = pd.read_csv(path, sep=r"\s+", header=None)
        df.columns = columns
    else:
        df = pd.DataFrame(columns=columns)  # Assign columns even if empty.
    dfs[key] = df

We now extract the arrays containing the occurrence time of each failure and the total dissipated energy in each event. In order to enable the possibility of filtering events in energy, we also produce a boolean mask to filter events with energy greater than a certain threshold. However, note that relating the dissipated energy with an observed outburst energy in the X-ray band is not straigtfarward. This is particularly true because during these failure events most of the energy could be converted into neutrino emission. Moreover, also less energetic events might produce short flares that could be detectable. Therefore, to be conservative at the moment, we consider all failure events regardless of their energy.

In [ ]:
t = {}
E = {}

mask_E = {}

for key, df in dfs.items():
    t[key] = df["time"].values
    E[key] = df["energy"].values
    mask_E[key] = E[key] > 0
    t[key] = t[key][mask_E[key]]
    E[key] = E[key][mask_E[key]]

In [ ]:
# Define initial magnetic fields at which we evaluate the interpolated failure rate curves.
# Note that this is needed to set the right colors.
log_B0_eval = np.linspace(11.0, 16.0, 100)

# Combine the arrays to find the global min and max values.
combined_values = np.concatenate([log_B0, log_B0_eval])
vmin, vmax = combined_values.min(), combined_values.max()

# Create a colormap and normalize it.
cmap = plt.cm.viridis
norm = Normalize(vmin=vmin, vmax=vmax)

Plot the failure rate from the simulation as a function of time and color-code them according to their initial magnetic field strength.

In [ ]:
# Define bin edges.
t_edges = np.logspace(0.0, 7, 20)
bin_widths = np.diff(t_edges)
bin_centers = t_edges[:-1] + bin_widths / 2

keys = dfs.keys()

fig, ax = plt.subplots(figsize=(15, 8))

rates = {}  # Store results for later use.

for i, key in enumerate(keys):
    # Compute histogram.
    counts, _ = np.histogram(t[key], bins=t_edges)

    # Convert to rate.
    rate = counts / bin_widths
    rates[key] = rate

    # Plot.
    ax.plot(
        bin_centers,
        rate,
        color=cmap(norm(log_B0[i])),
        rasterized=True,
        lw=4,
        alpha=1,
    )

# Redefine colorbar (unchanged).
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\log_{10}(B_0 \, {\rm[G]})$")

# Axes labels and scales.
ax.set_xlabel("time [yr]")
ax.set_ylabel("Failures per year")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(0.0001, 2000)

## Construct the interpolator function

In [ ]:
# Create a grid of initial magnetic fields and stack all the cooling curves together.
B0 = 10**log_B0
rate_t_stack = np.vstack([rates[key] for key in keys]).T

In [ ]:
# Define the minimum and maximum age in [yr] and the minimum and maximum initial magnetic field in [G].
time_range = np.array([0.0, 1.0e7])
B0_range = np.array([1.0e11, 1.0e16])

failure_rate_interpolator = interpolate.RectBivariateSpline(
    bin_centers,
    B0,
    rate_t_stack,
    bbox=[time_range[0], time_range[1], B0_range[0], B0_range[1]],
    kx=1,
    ky=1,
)

In [ ]:
# Save the interpolator function and try to import it again to see if it works.
interpolator_path = base_path.joinpath(
    cfg["magneto-thermal_path"], "interpolator_crust_failure_rate.pkl"
)
with open(
    interpolator_path,
    "wb",
) as f:
    pickle.dump(failure_rate_interpolator, f)

with open(
    interpolator_path,
    "rb",
) as f:
    failure_rate_interpolator_import = pickle.load(f)

In [ ]:
# Define a grid of times and initial magnetic fields at which we evaluate the interpolated cooling curves.
t_eval = np.logspace(0.0, 7, 500)
B0_eval = 10**log_B0_eval

failure_rate_interp = failure_rate_interpolator_import(t_eval, B0_eval)
print(failure_rate_interp.shape)

Plot the interpolated results as well as the smoothed curves as a consistency check.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(1.0e-5, 2.0e3)
ax.set_xlabel(r"Time [yr]")
ax.set_ylabel("Failures per year")

for i, key in enumerate(keys):
    ax.plot(
        bin_centers,
        rates[key],
        lw=4,
        alpha=1,
        color=cmap(norm(log_B0[i])),
        rasterized=True,
    )

for i in range(len(B0_eval)):
    ax.plot(
        t_eval,
        failure_rate_interp[:, i],
        linestyle="--",
        linewidth=4,
        color=cmap(norm(log_B0_eval[i])),
        rasterized=True,
        alpha=0.5,
    )

# Create a legend for the line styles.
styleandles = [
    Line2D(
        [0], [0], color="black", linestyle="-", linewidth=4, label="Original"
    ),
    Line2D(
        [0],
        [0],
        color="black",
        linestyle="--",
        linewidth=4,
        label="Interpolated",
    ),
]
plt.legend(handles=styleandles, frameon=False, loc=0)

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\log_{10}(B_0 \, {\rm[G]})$")

plt.grid()

In [ ]:
# Try to evaluate the failure rate for a set of random values of ages and initial magnetic fields.
t_eval_test = np.array([1.0e3, 1.0e2])
B0_eval_test = np.logspace(13.0, 15, 2)

failure_rate_interp_test = failure_rate_interpolator_import.ev(
    t_eval_test, B0_eval_test
)
print(failure_rate_interp_test)

Plot the magnetic energy released during the failures as a function of time.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

for i, key in enumerate(keys):
    ax.plot(
        t[key],
        E[key],
        linestyle="None",
        marker="o",
        color=cmap(norm(log_B0[i])),
        markersize=6,
        alpha=0.2,
        rasterized=True,
    )

# Labels and scales.
ax.set_xlabel("Time [yr]")
ax.set_ylabel("Crustal failure energy [erg]")
ax.set_xscale("log")
ax.set_yscale("log")

# Colorbar.
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\log_{10}(B_0 \, {\rm[G]})$")

plt.grid()
plt.show()

In [ ]:
# Compute the released total power of the failures as a function of time.
power = {}

for key in keys:
    energy, _ = np.histogram(t[key], bins=t_edges, weights=E[key])
    power[key] = (
        energy / bin_widths
    )  # Divide by bin width to obtain power per unit time.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

for i, key in enumerate(keys):
    ax.plot(
        bin_centers,
        power[key],
        color=cmap(norm(log_B0[i])),
        rasterized=True,
        lw=4,
        alpha=1,
    )

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\log_{10}(B_0 \, {\rm[G]})$")

ax.set_xlabel("time [yr]")
ax.set_ylabel(r"Failure power [erg s$^{-1}$]")
ax.set_xscale("log")
ax.set_yscale("log")

plt.show()

## 